# GIT

In [1]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer

import os
import re
import joblib
import numpy as np
import pandas as pd
from typing import List, Dict, Any

from ast import literal_eval

from src.models.NER_classifier import NERClassifier
from src.utils.paths import WEIGHTS_DIR

DEVICE = "cuda:1" if torch.cuda.is_available() and torch.cuda.device_count() > 1 else "cuda" if torch.cuda.is_available() else "cpu"
BASE_MODEL = "cointegrated/rubert-tiny2"

/home/smirnov@dohod.local/Desktop/x5/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def ner_pipeline(
    samples: List[Dict[str, Any]],
    models: Dict[str, torch.nn.Module],
    tokenizer,
    max_len: int = 64
) -> pd.DataFrame:
    all_results = []

    for item in samples:
        text = item["text"]
        words = text.split()

        encoding = tokenizer(
            words,
            is_split_into_words=True,
            return_tensors="pt",
            padding='max_length',
            truncation=True,
            max_length=max_len
        ).to(DEVICE)

        sample_result = {"sample": text}

        for model_name, model in models.items():
            with torch.no_grad():
                logits = model(encoding["input_ids"], encoding["attention_mask"])
                probs = F.softmax(logits, dim=-1)

            word_ids = encoding.word_ids(batch_index=0)
            current_word = None
            curr_1, curr_2 = [], []
            word_probs_1, word_probs_2 = [], []

            for w_id, p in zip(word_ids, probs.squeeze().cpu().numpy()):
                if w_id is None:
                    continue
                if w_id != current_word:
                    if current_word is not None:
                        word_probs_1.append(float(np.max(curr_1)))
                        word_probs_2.append(float(np.max(curr_2)))
                    current_word = w_id
                    curr_1, curr_2 = [p[1]], [p[2] if len(p) > 2 else 0.0]
                else:
                    curr_1.append(p[1])
                    curr_2.append(p[2] if len(p) > 2 else 0.0)

            if curr_1:
                word_probs_1.append(float(np.max(curr_1)))
                word_probs_2.append(float(np.max(curr_2)))

            sample_result[f"{model_name}_proba_1"] = word_probs_1
            sample_result[f"{model_name}_proba_2"] = word_probs_2

        all_results.append(sample_result)

    return pd.DataFrame(all_results)


In [3]:
VOLUME_RE = re.compile(r"(?<!\w)(\d+[.,]?\d*)\s?(л|л\.|литр\w*|мл|ml|г|кг|шт|уп\w*|пак\w*|бут\w*)(?!\w)", re.IGNORECASE)
PERCENT_RE = re.compile(r"(?<!\w)(\d+[.,]?\d*)\s?%|процент\w*", re.IGNORECASE)

def rule_spans(text):
    return [(m.start(), m.end(), "VOLUME") for m in VOLUME_RE.finditer(text)] + \
           [(m.start(), m.end(), "PERCENT") for m in PERCENT_RE.finditer(text)]

def apply_priors(text, words, probs, beta=2.0):
    offsets, char_idx = [], 0
    for w in words:
        start, end = text.find(w, char_idx), text.find(w, char_idx) + len(w)
        offsets.append((start, end))
        char_idx = end + 1

    for s, e, label in rule_spans(text):
        for i, (w_start, w_end) in enumerate(offsets):
            if max(s, w_start) < min(e, w_end):
                if label == "VOLUME":
                    probs[i][4] += beta
                elif label == "PERCENT":
                    probs[i][2] += beta
    return probs

def extract_features(row, words):
    n = len(words)
    features = []
    for i, word in enumerate(words):
        get_probs = lambda idx: [
            row['brand_probs'][idx],
            row['type_probs'][idx],
            row['percent_probs'][idx],
            row['volume_probs'][idx],
            row['o_probs'][idx],
        ] if 0 <= idx < n else [0]*5

        feat = get_probs(i) + [i, len(word)] + get_probs(i-1) + [i > 0] + get_probs(i+1) + [i < n-1]
        features.append(feat)
    return np.array(features, dtype=np.float32)

def annotate_sample(text, words, labels):
    annotations, prev_label, char_idx = [], None, 0
    for word, label in zip(words, labels):
        start, end = text.find(word, char_idx), text.find(word, char_idx) + len(word)
        bio = 'O' if label == 'O' else ('I-' + label if prev_label == label else 'B-' + label)
        annotations.append((start, end, bio))
        prev_label, char_idx = label, end + 1
    return annotations

def infer_ensemble(models, df, le):
    all_annots = []
    for _, row in df.iterrows():
        text, words = row['sample'], row['sample'].split()
        X = extract_features(row, words)
        probs = np.mean([m.predict_proba(X) for m in models], axis=0)
        probs = apply_priors(text, words, probs, beta=1.0)
        labels = le.inverse_transform(probs.argmax(axis=1))
        all_annots.append(annotate_sample(text, words, labels))
    df['annotation'] = all_annots
    return df

def boost_pipeline(df: pd.DataFrame) -> pd.DataFrame:
    for col in df.columns:
        if col != 'sample':
            df[col] = df[col].apply(lambda x: literal_eval(x) if isinstance(x, str) else x)

    xgboost_folder = os.path.join(WEIGHTS_DIR, "xgboost")
    models = [joblib.load(os.path.join(xgboost_folder, f"xgb_fold{fold}.joblib")) for fold in range(1, 6)]
    le = joblib.load(os.path.join(xgboost_folder, "label_encoder.joblib"))

    return infer_ensemble(models, df, le)

In [4]:
def initialize_models():

    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

    models = {
        "brand": NERClassifier(base_model_name=BASE_MODEL, num_labels=3, use_dropout=False),
        "type": NERClassifier(base_model_name=BASE_MODEL, num_labels=3, use_dropout=True),
        "volume": NERClassifier(base_model_name=BASE_MODEL, num_labels=3, use_dropout=False),
        "percent": NERClassifier(base_model_name=BASE_MODEL, num_labels=3, use_dropout=False),
        "o": NERClassifier(base_model_name=BASE_MODEL, num_labels=2, use_dropout=False),
    }

    for name, model in models.items():
        model_path = os.path.join(WEIGHTS_DIR, name, "model.pt")

        if not os.path.exists(model_path):
            raise FileNotFoundError(f"Model checkpoint not found: {model_path}")

        state_dict = torch.load(model_path, map_location=DEVICE)
        new_state_dict = {}
        for k, v in state_dict.items():
            new_k = k
            if new_k.startswith("base."):
                new_k = new_k.replace("base.", "base_model.")
            if new_k.startswith("cls."):
                new_k = new_k.replace("cls.", "classifier.")
            new_state_dict[new_k] = v

        model.load_state_dict(new_state_dict)
        model.to(DEVICE)
        model.eval()

    return models, tokenizer

In [5]:
df = pd.read_csv("data/raw/test.csv", delimiter=";")

samples = [{"text": row["sample"]} for _, row in df.iterrows()]
models, tokenizer = initialize_models()

ner_output = ner_pipeline(samples=samples, models=models, tokenizer=tokenizer)

submit = boost_pipeline(ner_output)
submit[['sample', 'annotation']].to_csv("xgboost_test_ensemble.csv", sep=";", index=False)

/usr/lib/python3.11/pickle.py:1718: UserWarning: [18:17:04] WARNING: /workspace/src/collective/../data/../common/error_msg.h:82: If you are loading a serialized model (like pickle in Python, RDS in R) or
configuration generated by an older version of XGBoost, please export the model by calling
`Booster.save_model` from that version first, then load it back in current version. See:

    https://xgboost.readthedocs.io/en/stable/tutorials/saving_model.html

for more details about differences between saving model and serializing.

  setstate(state)
/home/smirnov@dohod.local/Desktop/x5/.venv/lib/python3.11/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.7.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


KeyError: 'brand_probs'

In [6]:
ner_output

,sample,brand_proba_1,brand_proba_2,type_proba_1,type_proba_2,volume_proba_1,volume_proba_2,percent_proba_1,percent_proba_2,o_proba_1,o_proba_2
0,форма для выпечки,"[0.00013369008956942707, 3.052856118301861e-05...","[4.4002434151479974e-05, 1.63976783369435e-05,...","[0.9993703961372375, 0.0012589844409376383, 0....","[4.5800166844855994e-05, 0.205951526761055, 0....","[1.936886110343039e-05, 1.670196797931567e-05,...","[2.4679156922502443e-05, 9.273120667785406e-06...","[2.433305098747951e-06, 3.757505055546062e-06,...","[5.325684924173402e-06, 3.6665055631601717e-06...","[1.4209050277713686e-05, 0.5083915591239929, 0...","[0.0, 0.0, 0.0]"
1,фарш свиной,"[2.95009849651251e-05, 2.0781879356945865e-05]","[7.1726581154507585e-06, 4.323562279751059e-06]","[0.9978698492050171, 0.00011031502072000876]","[0.9684856534004211, 0.998604953289032]","[1.291603530262364e-05, 1.1735512089217082e-05]","[8.137678378261626e-06, 8.521326890331693e-06]","[2.2671067654300714e-06, 2.7361049887986155e-06]","[4.5765191316604614e-06, 3.616980620790855e-06]","[6.884136382723227e-06, 7.219177405204391e-06]","[0.0, 0.0]"
2,сок ананасовый без сахара,"[1.7093270798795857e-05, 8.502493983542081e-06...","[7.725037903583143e-06, 1.1647795872704592e-05...","[0.999487042427063, 0.0082270922139287, 0.0001...","[1.0684951121220365e-05, 0.9988677501678467, 0...","[7.244510925374925e-06, 1.1791326869570184e-05...","[1.2150751899753232e-05, 1.1365325917722657e-0...","[2.5231279323634226e-06, 4.357022589829285e-06...","[3.5282762382848887e-06, 4.715308477898361e-06...","[7.88328361522872e-06, 1.1249218005104922e-05,...","[0.0, 0.0, 0.0, 0.0]"
3,еринги,[0.0008616924751549959],[2.6674986202124273e-06],[0.9379611015319824],[0.02284291945397854],[2.7537565983948298e-05],[9.759520253282972e-06],[3.88903481507441e-06],[3.6232245292922016e-06],[0.057507116347551346],[0.0]
4,молооко,[0.0029691439121961594],[4.462505785340909e-06],[0.9816761016845703],[0.9973249435424805],[1.0131457202078309e-05],[1.1675823770929128e-05],[2.8201593522680923e-06],[3.7392539979919093e-06],[1.1617530617513694e-05],[0.0]
...,...,...,...,...,...,...,...,...,...,...,...
4995,milkywa,[0.9580310583114624],[4.082089799339883e-05],[0.9203284382820129],[0.8981713652610779],[2.4537659555790015e-05],[1.8716949853114784e-05],[5.0299449867452495e-06],[9.909845175570808e-06],[4.28903877036646e-06],[0.0]
4996,очиститель для унитаза,"[1.3232427590992302e-05, 1.4906081560184248e-0...","[3.0450646590907127e-06, 4.554816314339405e-06...","[0.9998018145561218, 6.36707991361618e-05, 0.0...","[0.9935799837112427, 0.0011680175084620714, 0....","[2.8646476494031958e-05, 2.6036872441181913e-0...","[2.0503444829955697e-05, 9.250635230273474e-06...","[3.3726344099704875e-06, 4.534600520855747e-06...","[5.317549039318692e-06, 4.616754267772194e-06,...","[1.1533324141055346e-05, 0.9999860525131226, 0...","[0.0, 0.0, 0.0]"
4997,арбузные,[0.00010377453872933984],[1.4011773146194173e-06],[0.9998218417167664],[0.9997736811637878],[1.0011215636041015e-05],[1.194961805595085e-05],[2.14135934584192e-06],[5.5543855523865204e-06],[5.713435712095816e-06],[0.0]
4998,кашы,[2.846535608114209e-05],[2.369680942138075e-06],[0.9994003772735596],[0.9989129304885864],[1.6074141967692412e-05],[1.5454768799827434e-05],[3.4403351492073853e-06],[2.77453023045382e-06],[4.022192570118932e-06],[0.0]
